# ActivationCache: bounded notebook display experiment

Synthetic CPU tensors; no model downloads. Run from this checkout with the locked dependencies. The largest allocation is 16 MiB; the largest single output is approximately 300,000 characters. Full tensor printing is restricted to four 4,096-element tensors.

Kernel formatting and `display()` return times **do not measure browser paint or responsiveness**. Observe scrolling and the final heartbeat separately. Results are written to `debugging/activation_cache_repr_measurements.json`.

**Execution status in this workspace:** kernel startup was blocked by ZeroMQ (`Operation not permitted`). The matrix was measured separately in in-process IPython; see the JSON results. Publication cells and browser responsiveness remain unmeasured. This notebook has no fabricated execution outputs.

In [ ]:
import gc
import inspect
import json
import platform
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from IPython.display import display
from transformer_lens import ActivationCache

root = Path.cwd()
if root.name == "debugging":
    root = root.parent
assert (root / "transformer_lens/ActivationCache.py").is_file()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from debugging.activation_cache_repr_probe import probe_cache

original_threads = torch.get_num_threads()
torch.set_num_threads(1)
original_print_options = vars(torch._tensor_str.PRINT_OPTS).copy()
results = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "activation_cache_blob": subprocess.check_output(["git", "hash-object", "transformer_lens/ActivationCache.py"], cwd=root, text=True).strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "class_file": inspect.getfile(ActivationCache),
    "device": "cpu", "dtype": "float32", "torch_threads": 1,
    "method": "One warm-up and five measured samples per probe operation; independently allocated all-ones tensors. Construction is timed once without warm-up and excluded from formatting samples. No inference or CUDA work.",
    "matrix": [], "publication": [],
    "browser_rendering": "Pending: kernel and captured-publication timings are not browser timings.",
}
print({key: results[key] for key in ["source_commit", "python", "torch", "device", "dtype"]})

In [ ]:
cases = [
    ("keys_1", 1, 1, "default", False),
    ("keys_100", 100, 1, "default", False),
    ("keys_10000", 10000, 1, "default", False),
    ("tensor_256", 4, 256, "default", True),
    ("tensor_4096", 4, 4096, "default", True),
    ("tensor_1048576", 4, 1048576, "default", True),
    ("tensor_4096_full", 4, 4096, "full", True),
]
try:
    for name, keys, elements, profile, include_values in cases:
        torch.set_printoptions(profile=profile)
        started = time.perf_counter()
        cache = ActivationCache({f"blocks.{i}.hook_resid_pre": torch.ones(elements) for i in range(keys)}, model=None)
        construction_ms = (time.perf_counter() - started) * 1000
        report = probe_cache(cache, repeats=5, include_values=include_values)
        assert all("error" not in item for item in report["measurements"].values()), report
        results["matrix"].append({"case": name, "elements_per_tensor": elements, "profile": profile, "construction_ms_single_sample": construction_ms, "report": report})
        selected = report["measurements"]
        print(name, "repr_ms=", round(selected["repr_wrapper"]["median_ms"], 4), "format_ms=", round(selected["ipython_format_wrapper"]["median_ms"], 4), "repr_chars=", selected["repr_wrapper"]["text_chars"])
        del cache
        gc.collect()
finally:
    torch.set_printoptions(**original_print_options)

In [ ]:
from IPython.utils.capture import capture_output

results["captured_display_matrix"] = []
try:
    for profile in ["default", "full"]:
        torch.set_printoptions(profile=profile)
        cache = ActivationCache({f"blocks.{i}.hook_resid_pre": torch.ones(4096) for i in range(4)}, model=None)
        for name, obj in [("wrapper", cache), ("dictionary", cache.cache_dict), ("tensor", next(iter(cache.values())))]:
            samples = []
            for iteration in range(6):
                started = time.perf_counter()
                with capture_output() as captured:
                    display(obj)
                elapsed_ms = (time.perf_counter() - started) * 1000
                if iteration:
                    samples.append(elapsed_ms)
            chars = sum(len(output.data.get("text/plain", "")) for output in captured.outputs)
            results["captured_display_matrix"].append({"profile": profile, "object": name, "samples_ms": samples, "median_ms": statistics.median(samples), "text_chars": chars})
            print(profile, name, "captured_display_ms=", round(statistics.median(samples), 4), "characters=", chars)
finally:
    torch.set_printoptions(**original_print_options)
    del cache


## Real kernel publication

The following cells publish actual IOPub display messages rather than capture them to memory. Each call is measured once, including first-call overhead. The measured interval ends when `display()` returns; the browser may still be processing the message. These are synthetic display cases, not a reproduction of the unlocated issue.

In [ ]:
def publish_case(label, obj):
    started = time.perf_counter()
    display(obj)
    elapsed_ms = (time.perf_counter() - started) * 1000
    results["publication"].append({"case": label, "display_return_ms_single_sample": elapsed_ms})
    print(f"{label}: display returned after {elapsed_ms:.3f} ms")

try:
    torch.set_printoptions(profile="default")
    cache = ActivationCache({f"blocks.{i}.hook_resid_pre": torch.ones(4096) for i in range(4)}, model=None)
    publish_case("wrapper_default", cache)
    publish_case("dictionary_default", cache.cache_dict)
    publish_case("tensor_default", next(iter(cache.values())))
    torch.set_printoptions(profile="full")
    publish_case("dictionary_full", cache.cache_dict)
finally:
    torch.set_printoptions(**original_print_options)
print("HEARTBEAT: bounded tensor publication finished")

In [ ]:
large_key_cache = ActivationCache({f"blocks.{i}.hook_resid_pre": torch.ones(1) for i in range(10000)}, model=None)
publish_case("wrapper_10000_keys", large_key_cache)
print("HEARTBEAT: large-key publication finished")
del cache, large_key_cache
gc.collect()

In [ ]:
output_path = root / "debugging/activation_cache_repr_measurements.json"
output_path.write_text(json.dumps(results, indent=2, allow_nan=False) + "\n")
torch.set_printoptions(**original_print_options)
torch.set_num_threads(original_threads)
print("EXPERIMENT COMPLETE: formatting and kernel publication measured; browser rendering is a separate observation.")